# My DIY Portfolio vs. the S&P 500

**What this notebook does:** You pick 5 stocks you like/believe in, and we build an equal-weight portfolio out of them. Then we compare it against just buying an S&P 500 index fund (SPY) over the same time period — same starting money, no cheating. Along the way we also make a **risk vs. return chart** so you can see which stocks were actually "worth" the risk you took on them, and calculate some real metrics real investors use (annualized return, volatility, Sharpe ratio, max drawdown).

**Why this matters:** Everyone *thinks* they can beat the market by picking good stocks. This notebook lets you actually test that idea with real historical data instead of just guessing.

**Level:** Built for a high-school student who knows what stocks/ETFs are and wants to go beyond "just plot the price" — but every step is explained in plain English.


In [ ]:
# Cell 1: Install and import everything we need
# yfinance = free stock price data, pandas/numpy = data handling, matplotlib = charts

!pip install yfinance --quiet

import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

# Make charts look a little nicer by default
plt.rcParams['figure.figsize'] = (11, 6)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

print("Libraries loaded. Ready to go.")


## Step 1: Pick your stocks

Change the tickers below to whatever companies you actually want to invest in (hypothetically). We're using 5 well-known names as an example — swap them for your own picks.

We'll compare this basket against **SPY**, which is an ETF that tracks the S&P 500 (basically "the market").


In [ ]:
# Cell 2: Set up your portfolio here — this is the only cell you really NEED to edit

MY_STOCKS_INPUT = "AAPL, MSFT, NVDA, KO, DIS"  #@param {type:"string"}
MY_STOCKS = [t.strip().upper() for t in MY_STOCKS_INPUT.split(",")]
BENCHMARK = "SPY"  #@param {type:"string"}

START_DATE = "2019-01-01"  #@param {type:"date"}
END_DATE = "2024-12-31"  #@param {type:"date"}

STARTING_MONEY = 10000  #@param {type:"number"}

print(f"Your portfolio: {MY_STOCKS}")
print(f"Benchmark: {BENCHMARK}")
print(f"Time period: {START_DATE} to {END_DATE}")


## Step 2: Download real price data

We pull daily historical prices from Yahoo Finance for every stock plus the benchmark. Some tickers occasionally have missing days (holidays, data gaps) — we handle that by forward-filling small gaps and dropping anything still broken.


In [ ]:
# Cell 3: Download historical price data

all_tickers = MY_STOCKS + [BENCHMARK]

raw_data = yf.download(all_tickers, start=START_DATE, end=END_DATE, auto_adjust=True, progress=False)

# yfinance returns a multi-level column structure when downloading multiple tickers —
# we just want the 'Close' price for each one
if isinstance(raw_data.columns, pd.MultiIndex):
    prices = raw_data['Close'].copy()
else:
    prices = raw_data[['Close']].copy()
    prices.columns = all_tickers

# Handle missing data: forward-fill small gaps (e.g. a ticker didn't trade one day),
# then drop any row that's still missing data at the start/end
prices = prices.ffill().dropna()

missing_tickers = [t for t in all_tickers if t not in prices.columns or prices[t].isna().all()]
if missing_tickers:
    print(f"Warning: could not get clean data for {missing_tickers} — check the ticker symbols.")

print(f"Downloaded {len(prices)} trading days of data.")
prices.tail()


## Step 3: Build the two portfolios

- **My Portfolio**: $10,000 split equally across your 5 stocks on day 1, then just held (no rebalancing — realistic for a "buy and forget" comparison).
- **Benchmark**: The same $10,000 put into SPY on day 1.

We track the value of both, day by day.


In [ ]:
# Cell 4: Turn prices into portfolio dollar values over time

# Daily returns for every ticker
daily_returns = prices.pct_change().dropna()

# --- My Portfolio: equal-weight across MY_STOCKS ---
my_stock_returns = daily_returns[MY_STOCKS]
portfolio_daily_return = my_stock_returns.mean(axis=1)   # equal weight = simple average each day
portfolio_value = STARTING_MONEY * (1 + portfolio_daily_return).cumprod()
portfolio_value.iloc[0] = STARTING_MONEY  # anchor the very first day to the starting amount

# --- Benchmark: 100% SPY ---
benchmark_daily_return = daily_returns[BENCHMARK]
benchmark_value = STARTING_MONEY * (1 + benchmark_daily_return).cumprod()
benchmark_value.iloc[0] = STARTING_MONEY

comparison = pd.DataFrame({
    "My Portfolio": portfolio_value,
    f"{BENCHMARK} (Index)": benchmark_value
})

print(f"Final value of My Portfolio: ${comparison['My Portfolio'].iloc[-1]:,.2f}")
print(f"Final value of {BENCHMARK} (Index): ${comparison[f'{BENCHMARK} (Index)'].iloc[-1]:,.2f}")


In [ ]:
# Cell 5: Chart — did your picks beat just buying the index?

fig, ax = plt.subplots()
ax.plot(comparison.index, comparison["My Portfolio"], label="My Portfolio", linewidth=2, color="#2563eb")
ax.plot(comparison.index, comparison[f"{BENCHMARK} (Index)"], label=f"{BENCHMARK} (Index Fund)",
        linewidth=2, color="#6b7280", linestyle="--")

ax.set_title(f"${STARTING_MONEY:,} Invested: My Portfolio vs. {BENCHMARK}", fontsize=14, fontweight="bold")
ax.set_xlabel("Date")
ax.set_ylabel("Portfolio Value ($)")
ax.legend(fontsize=11)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))

plt.tight_layout()
plt.show()


## Step 4: Risk vs. Return — was it worth it?

Beating the market means nothing if you took on way more risk to do it. This chart plots **each individual stock** (plus your portfolio and the benchmark) by:
- **X-axis**: volatility (how much the stock bounces around — a rough measure of risk)
- **Y-axis**: annualized return (how much you actually made per year)

Stocks in the top-left are the dream (high return, low risk). Bottom-right is the trap (high risk, low return).


In [ ]:
# Cell 6: Calculate risk and return for every stock, your portfolio, and the benchmark

TRADING_DAYS_PER_YEAR = 252

def annualized_return(daily_ret_series):
    total_growth = (1 + daily_ret_series).prod()
    n_years = len(daily_ret_series) / TRADING_DAYS_PER_YEAR
    return total_growth ** (1 / n_years) - 1

def annualized_volatility(daily_ret_series):
    return daily_ret_series.std() * np.sqrt(TRADING_DAYS_PER_YEAR)

def sharpe_ratio(daily_ret_series, risk_free_rate=0.04):
    # Sharpe ratio = extra return you got per unit of risk, vs. a "safe" ~4% risk-free rate (T-bill-ish)
    ann_ret = annualized_return(daily_ret_series)
    ann_vol = annualized_volatility(daily_ret_series)
    return (ann_ret - risk_free_rate) / ann_vol if ann_vol != 0 else np.nan

def max_drawdown(value_series):
    # Biggest drop from a peak to a later low -- "how bad could it have felt" at the worst moment
    running_max = value_series.cummax()
    drawdown = (value_series - running_max) / running_max
    return drawdown.min()

results = []

for ticker in MY_STOCKS:
    ret = daily_returns[ticker]
    val = STARTING_MONEY * (1 + ret).cumprod()
    results.append({
        "Name": ticker,
        "Annual Return": annualized_return(ret),
        "Annual Volatility": annualized_volatility(ret),
        "Sharpe Ratio": sharpe_ratio(ret),
        "Max Drawdown": max_drawdown(val)
    })

# Portfolio and benchmark too
results.append({
    "Name": "My Portfolio",
    "Annual Return": annualized_return(portfolio_daily_return),
    "Annual Volatility": annualized_volatility(portfolio_daily_return),
    "Sharpe Ratio": sharpe_ratio(portfolio_daily_return),
    "Max Drawdown": max_drawdown(portfolio_value)
})
results.append({
    "Name": f"{BENCHMARK} (Index)",
    "Annual Return": annualized_return(benchmark_daily_return),
    "Annual Volatility": annualized_volatility(benchmark_daily_return),
    "Sharpe Ratio": sharpe_ratio(benchmark_daily_return),
    "Max Drawdown": max_drawdown(benchmark_value)
})

results_df = pd.DataFrame(results).set_index("Name")

# Nicely formatted table
display_df = results_df.copy()
display_df["Annual Return"] = (display_df["Annual Return"] * 100).round(2).astype(str) + "%"
display_df["Annual Volatility"] = (display_df["Annual Volatility"] * 100).round(2).astype(str) + "%"
display_df["Sharpe Ratio"] = display_df["Sharpe Ratio"].round(2)
display_df["Max Drawdown"] = (display_df["Max Drawdown"] * 100).round(2).astype(str) + "%"

display_df


In [ ]:
# Cell 7: Chart -- risk vs. return scatter plot

fig, ax = plt.subplots(figsize=(10, 7))

for name, row in results_df.iterrows():
    is_special = name in ["My Portfolio", f"{BENCHMARK} (Index)"]
    color = "#2563eb" if name == "My Portfolio" else "#6b7280" if name == f"{BENCHMARK} (Index)" else "#93c5fd"
    size = 220 if is_special else 140
    marker = "*" if is_special else "o"

    ax.scatter(row["Annual Volatility"] * 100, row["Annual Return"] * 100,
               s=size, color=color, marker=marker, edgecolors="black", linewidth=0.8, zorder=3)
    ax.annotate(name, (row["Annual Volatility"] * 100, row["Annual Return"] * 100),
                textcoords="offset points", xytext=(8, 6), fontsize=10)

ax.axhline(0, color="gray", linewidth=0.8)
ax.set_xlabel("Annualized Volatility / Risk (%)", fontsize=12)
ax.set_ylabel("Annualized Return (%)", fontsize=12)
ax.set_title("Risk vs. Return: Which Stocks Were Actually Worth It?", fontsize=14, fontweight="bold")

plt.tight_layout()
plt.show()

print("Top-left = best (high return, low risk). Bottom-right = worst (high risk, low return).")


## Step 5: Drawdown chart — the scariest moment to hold your portfolio

"Max drawdown" answers: *at the worst point, how much would you have watched your money shrink from its peak?* This is often more emotionally important than average returns — it's the number that tests whether you'd have actually held on or panic-sold.


In [ ]:
# Cell 8: Chart -- drawdown over time for My Portfolio vs. Benchmark

def drawdown_series(value_series):
    running_max = value_series.cummax()
    return (value_series - running_max) / running_max * 100

port_dd = drawdown_series(portfolio_value)
bench_dd = drawdown_series(benchmark_value)

fig, ax = plt.subplots()
ax.fill_between(port_dd.index, port_dd, 0, color="#2563eb", alpha=0.4, label="My Portfolio")
ax.fill_between(bench_dd.index, bench_dd, 0, color="#6b7280", alpha=0.4, label=f"{BENCHMARK} (Index)")

ax.set_title("Drawdowns: How Far Below the Peak Did We Fall?", fontsize=14, fontweight="bold")
ax.set_ylabel("Drawdown (%)")
ax.set_xlabel("Date")
ax.legend(fontsize=11)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))

plt.tight_layout()
plt.show()


## Summary

This notebook built a hypothetical $10,000 portfolio from 5 chosen stocks and compared it head-to-head against simply buying an S&P 500 index fund (SPY) over the same period — tracking final value, annualized return, volatility, Sharpe ratio, and max drawdown for every position. It also visualized a risk-vs-return scatter plot to show which individual stocks were actually worth their risk, and a drawdown chart to show the roughest stretch an investor would have had to sit through.

**Resume-ready bullet:**
*"Built a Python-based portfolio analysis tool comparing a custom 5-stock equity portfolio against the S&P 500 benchmark, calculating annualized return, volatility, Sharpe ratio, and maximum drawdown using real historical market data (yfinance)."*
